# Pathway analysis

In [1]:
import scanpy as sc
import decoupler as dc

# Only needed for processing
import numpy as np
import pandas as pd

In [2]:
comparison = 'age'
experiment = "feature_importance_smote_catboost_7b_gene_ranking_07"


In [3]:
dds = pd.read_csv(f'/home/amore/work/data/feature_importance_smote_catboost_7b_gene_ranking_07.csv',index_col=0)
dds

,Ensembl,coef,abs_coef
Symbol,,,
NT5C2,ENSG00000076685.18,1.413563e+01,1.413563e+01
AC012146.4,ENSG00000262855.1,1.204186e+01,1.204186e+01
UBFD1,ENSG00000103353.15,1.083837e+01,1.083837e+01
ALDOA,ENSG00000285043.1,7.641741e+00,7.641741e+00
H3F3B,ENSG00000132475.10,6.241443e+00,6.241443e+00
...,...,...,...
RSPRY1,ENSG00000159579.13,5.070000e-06,5.070000e-06
RNA5SP385,ENSG00000251756.1,2.550000e-06,2.550000e-06
TRIM24,ENSG00000122779.17,8.830000e-07,8.830000e-07


In [4]:
results_df =dds

## Get msigdb

In [5]:
msigdb = pd.read_csv('msigdb.csv')
#msigdb = dc.get_resource('MSigDB')
msigdb# Sort the results by the ranking metric (e.g., log2 fold change)


,Unnamed: 0,genesymbol,collection,geneset
0,0,MAFF,chemical_and_genetic_perturbations,BOYAULT_LIVER_CANCER_SUBCLASS_G56_DN
1,1,MAFF,chemical_and_genetic_perturbations,ELVIDGE_HYPOXIA_UP
2,2,MAFF,chemical_and_genetic_perturbations,NUYTTEN_NIPP1_TARGETS_DN
3,3,MAFF,immunesigdb,GSE17721_POLYIC_VS_GARDIQUIMOD_4H_BMDC_DN
4,4,MAFF,chemical_and_genetic_perturbations,SCHAEFFER_PROSTATE_DEVELOPMENT_12HR_UP
...,...,...,...,...
3825623,3838543,PRAMEF22,go_biological_process,GOBP_POSITIVE_REGULATION_OF_CELL_POPULATION_PR...
3825624,3838544,PRAMEF22,go_biological_process,GOBP_APOPTOTIC_PROCESS
3825625,3838545,PRAMEF22,go_biological_process,GOBP_REGULATION_OF_CELL_DEATH
3825626,3838546,PRAMEF22,go_biological_process,GOBP_NEGATIVE_REGULATION_OF_DEVELOPMENTAL_PROCESS


In [6]:
# Filter by hallmark
#msigdb = msigdb[msigdb['collection']=='hallmark']

# Remove duplicated entries
msigdb = msigdb[~msigdb.duplicated(['geneset', 'genesymbol'])]

# Rename
#msigdb.loc[:, 'geneset'] = [name.split('HALLMARK_')[1] for name in msigdb['geneset']]

msigdb

,Unnamed: 0,genesymbol,collection,geneset
0,0,MAFF,chemical_and_genetic_perturbations,BOYAULT_LIVER_CANCER_SUBCLASS_G56_DN
1,1,MAFF,chemical_and_genetic_perturbations,ELVIDGE_HYPOXIA_UP
2,2,MAFF,chemical_and_genetic_perturbations,NUYTTEN_NIPP1_TARGETS_DN
3,3,MAFF,immunesigdb,GSE17721_POLYIC_VS_GARDIQUIMOD_4H_BMDC_DN
4,4,MAFF,chemical_and_genetic_perturbations,SCHAEFFER_PROSTATE_DEVELOPMENT_12HR_UP
...,...,...,...,...
3825623,3838543,PRAMEF22,go_biological_process,GOBP_POSITIVE_REGULATION_OF_CELL_POPULATION_PR...
3825624,3838544,PRAMEF22,go_biological_process,GOBP_APOPTOTIC_PROCESS
3825625,3838545,PRAMEF22,go_biological_process,GOBP_REGULATION_OF_CELL_DEATH
3825626,3838546,PRAMEF22,go_biological_process,GOBP_NEGATIVE_REGULATION_OF_DEVELOPMENTAL_PROCESS


In [7]:
set(msigdb["collection"])

{'biocarta_pathways',
 'cancer_gene_neighborhoods',
 'cancer_modules',
 'cell_type_signatures',
 'chemical_and_genetic_perturbations',
 'go_biological_process',
 'go_cellular_component',
 'go_molecular_function',
 'hallmark',
 'human_phenotype_ontology',
 'immunesigdb',
 'kegg_pathways',
 'mirna_targets_legacy',
 'mirna_targets_mirdb',
 'oncogenic_signatures',
 'pid_pathways',
 'positional',
 'reactome_pathways',
 'tf_targets_gtrf',
 'tf_targets_legacy',
 'vaccine_response',
 'wikipathways'}

In [8]:
results_df.head()

,Ensembl,coef,abs_coef
Symbol,,,
NT5C2,ENSG00000076685.18,14.135628,14.135628
AC012146.4,ENSG00000262855.1,12.041862,12.041862
UBFD1,ENSG00000103353.15,10.838371,10.838371
ALDOA,ENSG00000285043.1,7.641741,7.641741
H3F3B,ENSG00000132475.10,6.241443,6.241443


## GSEA

In [9]:
# Prepare the ranking for GSEA
ranked_genes = results_df[['coef']].sort_values(by='coef', ascending=False)

# Prepare the ranking for GSEA
gene_list = ranked_genes['coef']
pd.DataFrame(gene_list)

,coef
Symbol,
NT5C2,1.413563e+01
AC012146.4,1.204186e+01
UBFD1,1.083837e+01
ALDOA,7.641741e+00
H3F3B,6.241443e+00
...,...
RSPRY1,5.070000e-06
RNA5SP385,2.550000e-06
TRIM24,8.830000e-07


In [10]:
gene_list = gene_list[gene_list.index.notnull()]
gene_list = pd.DataFrame(gene_list)
gene_list

,coef
Symbol,
NT5C2,1.413563e+01
AC012146.4,1.204186e+01
UBFD1,1.083837e+01
ALDOA,7.641741e+00
H3F3B,6.241443e+00
...,...
RSPRY1,5.070000e-06
RNA5SP385,2.550000e-06
TRIM24,8.830000e-07


In [11]:
mitocarta_data = pd.read_csv('../data/mitoCarta_terms_tissues_only.csv')
mitocarta_data['collection'] = 'mitocarta'
mitocarta_subset = mitocarta_data[['Symbol', 'collection']]
msigdb_extended = pd.concat([msigdb, mitocarta_subset], ignore_index=True)
msigdb_extended = msigdb_extended[~msigdb_extended.duplicated(['geneset', 'genesymbol'])]


In [ ]:
# Run GSEA
enr_gsea = dc.get_gsea_df(
    df=gene_list,
    stat="coef",
    net=msigdb,
    source='geneset',
    target='genesymbol'
)

# Display the top enrichment results
enr_gsea.head()

In [ ]:
enr_gsea.to_csv(f'/home/amore/work/data/{experiment}_{comparison}_GSEA.csv', header=True)

In [ ]:
significant_GSEA = enr_gsea[enr_gsea["FDR p-value"] < 0.1].sort_values(by="NES")
significant_GSEA

## ORA

In [12]:
# Infer enrichment with ora using significant deg
#top_genes = results_df[results_df['padj'] < 0.05]

# Run ora
enr_pvals = dc.get_ora_df(
    df=gene_list.index,
    net=msigdb,
    source='geneset',
    target='genesymbol'
)

enr_pvals.head()

,Term,Set size,Overlap ratio,p-value,FDR p-value,Odds ratio,Combined score,Features
0,AAACCAC_MIR140,110,0.018182,0.582527,0.863081,1.273016,0.687913,NUTF2;PHF20L1
1,AAAGACA_MIR511,204,0.029412,0.154646,0.654223,1.800368,3.360595,CALM1;DEDD;MRPL21;SMARCE1;TRIM24;UBE2H
2,AAAGGAT_MIR501,127,0.031496,0.188298,0.691749,1.995498,3.331940,BCL6;KIF2A;PHF6;TOGARAM1
3,AAAGGGA_MIR204_MIR211,224,0.022321,0.364159,0.773651,1.382317,1.396369,ARAP2;CHP1;PRDM2;PRRX1;STXBP5
4,AAANWWTGC_UNKNOWN,194,0.030928,0.130967,0.639659,1.893888,3.849909,BCL6;CALM1;MAN2A2;MGLL;PPP2R2A;THAP12


In [ ]:
enr_pvals.to_csv(f'/home/amore/work/data/{experiment}_{comparison}_ORA.csv', header=True)

In [ ]:
enr_pvals= pd.read_csv(f'/home/amore/work/data/{experiment}_{comparison}_ORA.csv', index_col=0)
enr_pvals

In [ ]:
significant_ORA = enr_pvals[(enr_pvals["p-value"] < 0.05) & (enr_pvals["FDR p-value"] < 0.05)]
significant_ORA.sort_values(by="Combined score")

In [ ]:
dc.plot_dotplot(
    enr_pvals.sort_values('Combined score', ascending=False).head(15),
    x='Combined score',
    y='Term',
    s='Odds ratio',
    c='FDR p-value',
    scale=.15,
    figsize=(5, 10)
)

In [ ]:
from plot_GSEA_ORA import *
plot_ora_results(enr_pvals)

In [ ]:
fig, ax = plot_ora_results(enr_pvals, top_n=3, figsize=(10, 5), scale_odds_ratio=20, 
                     fontsize_title=16, fontsize_subtitle=14, fontsize_text=12,
                     fdr_bar_size=[0.8, 0.15, 0.03, 0.5], bbox_to_anchor=(1.3, 1.05))

In [ ]:
fig.savefig(f"results/{experiment}_{comparison}_ORA_bubble.png", bbox_inches='tight')


In [ ]:
set(msigdb["collection"])